# Colab quick start (all in one)

**Run all, once.** This notebook does only what needs the GPU: vLLM, the model (downloaded if you do not have it yet) and 30 training questions.
Results are saved to Google Drive. Everything else (tests, the fixture check, scoring, parameter count, licence check) is done on your own computer without a GPU.

Only one manual step: put **`train.zip`** (the training split you received, 128 MB) in `MyDrive/DataMate/` on Google Drive.
The model is an **example** (`Qwen/Qwen3-8B`): change `MODEL_REPO` to try another. Which model is used for the competition is the team lead's decision;
the license, release date and size must be checked on the model card (the notebook prints the parameter count).

In [ ]:
MODEL_REPO = "Qwen/Qwen3-8B"          # EXAMPLE model, Hugging Face repo id
VLLM_VERSION = "0.10.1.1"             # same vLLM as the reference base image (M4-01)
QUESTIONS  = 30                       # how many training questions to run
DRIVE      = "/content/drive/MyDrive/DataMate"

In [ ]:
# 1. Basics: Drive, GPU
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
T0 = time.time()
MODEL_NAME = MODEL_REPO.split("/")[-1]
def sh(cmd, tail=1500):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True); print(r.stdout[-tail:], r.stderr[-800:])
    assert r.returncode == 0, f"failed: {cmd}"; return r.stdout
ENV = {**os.environ, "PYTHONPATH": "/content/DataMate/src", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "PYTHONHASHSEED": "0"}
def run(args): return subprocess.run([PY, "-m", *args], cwd="/content/DataMate", env=ENV, text=True, capture_output=True)

from google.colab import drive
drive.mount("/content/drive"); Path(DRIVE).mkdir(parents=True, exist_ok=True)
gpu = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader").strip()
gpu_name, gpu_mem, cap = [x.strip() for x in gpu.split(",")]
print("GPU:", gpu_name, "| compute capability", cap, "" if cap == "12.0" else "(not the competition GPU type: speed and memory numbers are only indicative)")
assert float(cap) >= 8.0 and float(gpu_mem.split()[0]) / 1024 >= 22, (
    f"{gpu_name} ({gpu_mem}, cc {cap}) cannot serve the model: Runtime -> Change runtime type -> G4 (Blackwell, like the RTX 5090)")

In [ ]:
# 2. Project code
shutil.rmtree("/content/DataMate", ignore_errors=True)
sh("git clone -q https://github.com/Kyoyooo/DataMate /content/DataMate")
sys.path[:0] = ["/content/DataMate/tools"]
import colab_helpers
print("commit:", sh("cd /content/DataMate && git rev-parse --short HEAD").strip())

In [ ]:
# 3. vLLM (the model server) pinned to the base image version, in its own Python 3.12 environment
#    (Colab's Python is too new for it). First time: a few minutes.
PY, PYINFO = colab_helpers.ensure_vllm_env(VLLM_VERSION, Path("/content/DataMate"))
print(PYINFO)

In [ ]:
# 4. Training data: from Drive (MyDrive/DataMate/data/train/ if you have it, otherwise unzipped from MyDrive/DataMate/train.zip)
TRAIN = Path("/content/snap/train"); TRAIN.parent.mkdir(parents=True, exist_ok=True)
if not TRAIN.exists():
    folder, zip_path = Path(DRIVE) / "data/train", Path(DRIVE) / "train.zip"
    if folder.is_dir(): shutil.copytree(folder, TRAIN)
    else:
        assert zip_path.is_file(), f"Put train.zip into {DRIVE}/ on Google Drive, then run this cell again."
        with zipfile.ZipFile(zip_path) as z: z.extractall(TRAIN.parent)
assert (TRAIN / "answers.jsonl").is_file(), "this must be the TRAINING split (answers.jsonl present)"
print("training data:", TRAIN)

In [ ]:
# 5. Model: use the copy on Drive, or download it once (EXAMPLE: MODEL_REPO) and keep a copy on Drive for next time
MODEL = Path("/content/snap/models") / MODEL_NAME; DRIVE_MODEL = Path(DRIVE) / "models" / MODEL_NAME
source = "already on local disk"
if not MODEL.exists():
    MODEL.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_MODEL.is_dir(): shutil.copytree(DRIVE_MODEL, MODEL); source = "copied from Drive"
    else:
        source = "downloaded from Hugging Face"
        from huggingface_hub import snapshot_download
        snapshot_download(MODEL_REPO, local_dir=str(MODEL))
        shutil.copytree(MODEL, DRIVE_MODEL)
print("model:", MODEL_REPO, "|", source)

In [ ]:
# 6. Real run: 30 training questions. The model server starts automatically (memory limited to about 32 GB, like the competition GPU).
SUB = Path("/content/subset"); shutil.rmtree(SUB, ignore_errors=True)
print(colab_helpers.make_subset_split(TRAIN, SUB, 2, QUESTIONS), "questions selected")
OUT = Path("/content/out/subset"); shutil.rmtree(OUT, ignore_errors=True)
frac = round(min(0.95, 32 / (float(gpu_mem.split()[0]) / 1024)), 3)
sets = {"model.path": json.dumps(str(MODEL)), "serving.gpu_memory_utilization": frac, "work_dir": json.dumps("/content/work/subset"),
        "serving.vllm_command": json.dumps([PY, "-m", colab_helpers.VLLM_MODULE])}
args = ["datamate.run", "--dev", "--config", "config/colab.json", "--data-dir", str(SUB), "--result-dir", str(OUT)]
for k, v in sets.items(): args += ["--set", f"{k}={v}"]
r = run(args); print(r.stderr[-1500:])
assert r.returncode == 0, "run failed: see /content/out/subset/logs/replica-0.log"
print(f"done in {(time.time()-T0)/60:.1f} min. Results: {OUT}")

In [ ]:
# 7. Save the results to Drive
dest = Path(DRIVE) / "runs" / time.strftime("%Y-%m-%d-%H%M"); shutil.copytree(OUT, dest); shutil.copy(SUB / "answers.jsonl", dest / "answers.jsonl")
print("saved to", dest, "\nScore it on your own computer (no GPU needed): python -m datamate.eval.score --pred <dest>/submission.jsonl --answers <dest>/answers.jsonl")